# MetAgeFormer — 2. Metabolomic Age (DeepGompertz)

Load the pretrained backbone + finetuned DeepGompertz head and predict:

- `metabolomic_age` — metabolomic aging clock (years)
- `age_gap` — metabolomic_age − chronological age (age acceleration)
- `mortality_risk_10y` — 10-year all-cause mortality risk (auxiliary output)

**Requirements**
- `Model_Weights/MetAgeFormer/` (backbone: `config.json`, `tokenizer.pkl`, `model_weights.pth`)
- `Model_Weights/DeepGompertz/model_weights.pth` (finetuned head)
- An AnnData NMR dataset with layer `Z-score normalized` and obs age column
  `"Age at assessment (estimated)"`

The inference utilities come from `Src/finetune/deep_gompertz/eval_common.py`
(the same inference code used for the paper's evaluation).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "Src"))

import anndata as ad
import pandas as pd
import torch

from common.constants import AGE_COL
from finetune.deep_gompertz.eval_common import load_model, predict

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Load backbone + DeepGompertz head

In [ ]:
PRETRAINED_DIR = REPO_ROOT / "Model_Weights" / "MetAgeFormer"
HEAD_DIR = REPO_ROOT / "Model_Weights" / "DeepGompertz"

model, tokenizer, pretrained_config, head_config = load_model(
    str(PRETRAINED_DIR), str(HEAD_DIR), DEVICE
)
print("DeepGompertz head config:")
for k, v in head_config.items():
    print(f"  {k}: {v}")

## 2. Input data (AnnData NMR with age)

In [ ]:
DATA_PATH = REPO_ROOT / "Data" / "NMR_dataset_fullcohort_107nonderived" / "val.h5ad"
# Fake-data demo: DATA_PATH = REPO_ROOT / "Data" / "fake" / "NMR_dataset_fake" / "val.h5ad"

adata = ad.read_h5ad(DATA_PATH)
assert AGE_COL in adata.obs, f"obs must contain age column {AGE_COL!r}"
adata

## 3. Predict

In [ ]:
prediction, _ = predict(
    model,
    tokenizer,
    adata,
    data_layer="Z-score normalized",
    age_col=AGE_COL,
    device=DEVICE,
    batch_size=256,
    num_workers=0,
    prefetch_factor=2,
)
prediction.head()

**Columns**
- `chronological_age` — observed age (years)
- `metabolomic_age` — metabolomic aging clock
- `age_gap` — age acceleration (`metabolomic_age − chronological_age`)
- `mortality_risk_10y` — auxiliary: predicted 10-year mortality probability
- `linear_predictor` / `alpha_i` / `gamma_i` — Gompertz risk components

In [ ]:
summary = prediction[["chronological_age", "metabolomic_age", "age_gap", "mortality_risk_10y"]].describe()
summary

## 4. (Optional) Age clock plot

`matplotlib` is optional (`pip install matplotlib`).

In [ ]:
try:
    import matplotlib.pyplot as plt

    x = prediction["chronological_age"].to_numpy()
    y = prediction["metabolomic_age"].to_numpy()

    plt.figure(figsize=(5, 5))
    plt.scatter(x, y, s=8, alpha=0.6)
    lims = [min(x.min(), y.min()), max(x.max(), y.max())]
    plt.plot(lims, lims, "k--", lw=1, label="identity")
    plt.xlabel("Chronological age (years)")
    plt.ylabel("Metabolomic age (years)")
    plt.legend()
    plt.title("MetAgeFormer aging clock")
    plt.show()
except ImportError:
    print("matplotlib not installed; skipping plot")